# 10 · Under the hood: partitions, shuffles, plans & RDDs

You can write Spark without knowing this — but you can't write *fast* Spark, or debug a slow job,
without it.

## Jobs, stages and tasks
* Each **action** starts a **job**.
* A job is split into **stages** at every **shuffle** (see below).
* Each stage runs one **task per partition**, in parallel on the executors.

So: number of partitions = amount of parallelism.

In [ ]:
# Setup — run this cell first every time you open the notebook
import os, sys
os.environ.setdefault("JAVA_HOME", "/opt/homebrew/opt/openjdk@17")   # Spark runs on Java
os.environ["PYSPARK_PYTHON"] = sys.executable                         # workers use this same Python

from pyspark.sql import SparkSession, Window, functions as F, types as T

spark = (SparkSession.builder
         .appName("pyspark-course")
         .master("local[*]")                               # run locally, use all CPU cores
         .config("spark.sql.shuffle.partitions", "8")      # explained in lesson 10
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("FATAL")                   # keep Java's log noise out of the notebook

DATA, OUT = "../data", "../output"

In [ ]:
import io, contextlib

def plan_of(df):
    """Return the text that df.explain() would print."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

print("cores Spark is using (default parallelism):", spark.sparkContext.defaultParallelism)
df = spark.range(0, 1_000_000)
print("partitions:", df.rdd.getNumPartitions())
df.groupBy(F.spark_partition_id().alias("partition")).count().orderBy("partition").show()

## Changing the number of partitions
| | `repartition(n)` / `repartition("col")` | `coalesce(n)` |
|---|---|---|
| Can increase? | yes | no — only decrease |
| Shuffle? | **yes** (full shuffle, even sizes) | no — just merges neighbouring partitions |
| Use for | more parallelism, or co-locating rows by key | fewer output files at the end of a job |

In [ ]:
print(df.repartition(16).rdd.getNumPartitions())
print(df.coalesce(2).rdd.getNumPartitions())
print(df.coalesce(100).rdd.getNumPartitions(), "← coalesce can't increase")

## Narrow vs wide transformations — and the shuffle
* **Narrow** (`select`, `filter`, `withColumn`, …): each output partition depends on **one** input partition.
  No data moves between executors — very cheap.
* **Wide** (`groupBy`, `join`, `orderBy`, `distinct`, `repartition`, …): rows with the same key must meet
  on the same executor, so data is **shuffled** across the network (and written to disk in between).
  Shuffles are usually the most expensive part of a job.

```
narrow (filter)            wide (groupBy key)
P0 ──► P0'                 P0 ─┬─► K=a
P1 ──► P1'                 P1 ─┼─► K=b      ← every partition sends data to every other
P2 ──► P2'                 P2 ─┴─► K=c
```
In a plan a shuffle appears as **`Exchange`**.

In [ ]:
orders = spark.read.csv(f"{DATA}/orders.csv", header=True, inferSchema=True)
orders.filter(F.col("quantity") > 2).explain()                      # no Exchange
orders.groupBy("status").agg(F.sum("quantity")).explain()           # Exchange hashpartitioning(status)

## Reading a query plan
Read physical plans **bottom-up**: the bottom is the data source, the top is the final result.
`explain("formatted")` gives a tidier version; `explain(True)` shows every stage of the optimizer:

1. **Parsed / Analyzed logical plan** — what you asked for
2. **Optimized logical plan** — after Catalyst rewrites it (e.g. pushes filters down, prunes columns)
3. **Physical plan** — the actual operators that will run

In [ ]:
q = (orders.select("order_id", "status", "quantity")
           .withColumn("double", F.col("quantity") * 2)
           .filter(F.col("status") == "returned"))     # filter written *last*…
q.explain(True)                                         # …but the optimizer moves it before the projection

## `spark.sql.shuffle.partitions` and AQE
After a shuffle, Spark creates `spark.sql.shuffle.partitions` partitions (default **200** — we set 8 in
the setup cell because our data is tiny). Too many → lots of tiny tasks; too few → huge tasks that spill to disk.

**Adaptive Query Execution (AQE)**, on by default since Spark 3.2, re-optimises *while running* using
real statistics: it merges small shuffle partitions, switches to broadcast joins when a side turns out
small, and splits skewed partitions. That's why you'll see `AdaptiveSparkPlan` at the top of plans.

In [ ]:
print(spark.conf.get("spark.sql.shuffle.partitions"), spark.conf.get("spark.sql.adaptive.enabled"))

## RDDs — Spark's original API
Before DataFrames, everything was an **RDD** (Resilient Distributed Dataset): a distributed collection of
arbitrary Python objects, manipulated with functional operations. DataFrames are built on top of RDDs.

Today you should **use DataFrames** (they get the optimizer; RDD code is opaque Python, like a UDF),
and RDDs aren't available at all in *Spark Connect* (the newer client/server mode used by Databricks
serverless). But you'll meet RDDs in older code and in interviews, so here are the basics:

In [ ]:
sc = spark.sparkContext
rdd = sc.parallelize(range(1, 11), numSlices=4)
print(rdd.map(lambda x: x * x).filter(lambda x: x % 2 == 0).collect())
print(rdd.reduce(lambda a, b: a + b))

pairs = sc.parallelize([("a", 1), ("b", 1), ("a", 1), ("c", 1), ("a", 1)])
print(sorted(pairs.reduceByKey(lambda a, b: a + b).collect()))      # key-value aggregation

In [ ]:
text = ["the cat sat on the mat", "the dog sat on the log", "a cat and a dog"]
text_rdd = sc.parallelize(text)
print(text_rdd.flatMap(lambda line: line.split()).take(8))       # flatMap = map + flatten

### Exercise 1
**Word count with RDDs**: using `text_rdd`, create `rdd_top3`: a Python list of the 3 most common `(word, count)` tuples, sorted by count descending then word ascending.

*Hint:* `flatMap` → `map(lambda w: (w, 1))` → `reduceByKey` → `collect` → Python `sorted(..., key=lambda kv: (-kv[1], kv[0]))`.

In [ ]:
rdd_top3 = None  # TODO

In [ ]:
# Check — run me after your answer
assert rdd_top3 == [("the", 4), ("a", 2), ("cat", 2)], rdd_top3
print("Correct!")

### Exercise 2
**Word count with DataFrames**: from `text_df` (provided), create `df_wc` with columns **`word`, `count`**, sorted by count descending then word.

In [ ]:
text_df = spark.createDataFrame([(t,) for t in text], "line string")
df_wc = None  # TODO

In [ ]:
# Check — run me after your answer
assert df_wc.columns == ["word", "count"]
assert [tuple(r) for r in df_wc.take(3)] == [("the", 4), ("a", 2), ("cat", 2)]
print("Correct!")

### Exercise 3
Using the `plan_of(df)` helper defined at the top, set **`groupby_shuffles`** to `True`/`False` depending on whether `orders.groupBy("status").count()` shuffles, and **`filter_shuffles`** likewise for `orders.filter("quantity > 3")`. Compute them from the plan text, don't hard-code!

In [ ]:
groupby_shuffles = None  # TODO
filter_shuffles = None   # TODO

In [ ]:
# Check — run me after your answer
assert groupby_shuffles is True and filter_shuffles is False
print("Correct!")

### Exercise 4
Store in `n_parts` the number of partitions of `orders` after **repartitioning by `customer_id` into 5 partitions**.

In [ ]:
n_parts = None  # TODO

In [ ]:
# Check — run me after your answer
assert n_parts == 5
print("Correct!")

---
Stuck? Worked answers (with outputs) are in [`../solutions/10_under_the_hood.ipynb`](../solutions/10_under_the_hood.ipynb).